In [23]:
import json

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
from embedder import Embedder
from evaluation_utils import llm_structured

load_dotenv()

True

In [24]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

In [25]:
class Questions(BaseModel):
    questions: list[str]

In [26]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [27]:
client = OpenAI()

def generate_questions(doc):
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })
    parsed, usage = llm_structured(
        client, data_gen_instructions, user_prompt, Questions
    )
    return parsed, usage

In [28]:
first_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

docs_by_filename = {doc["filename"]: doc for doc in documents}

input_tokens = []
for filename in first_pages:
    parsed, usage = generate_questions(docs_by_filename[filename])
    input_tokens.append(usage.input_tokens)
    print(filename, usage.input_tokens)

01-agentic-rag/lessons/01-intro.md 1021
01-agentic-rag/lessons/02-environment.md 1287
01-agentic-rag/lessons/03-rag.md 1754


In [ ]:
# Q1. Generating questions
sum(input_tokens) / len(input_tokens)

In [ ]:
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth)

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

In [ ]:
embed = Embedder()

batch_size = 50
X = []
for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    X.extend(embed.encode_batch([chunk["content"] for chunk in batch]))
X = np.array(X)

In [ ]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    return vindex.search(embed.encode(query), num_results=num_results)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
# Q2. First result with text search
q = ground_truth[0]["question"]
text_search(q)[0]["filename"]

In [ ]:
# Q3. First result with vector search
vector_search(q)[0]["filename"]

In [ ]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])
    return [int(d["filename"] == filename) for d in results]

def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]

In [ ]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
# Q4. Evaluating text search
evaluate(ground_truth, text_search)

In [ ]:
# Q5. Evaluating vector search
evaluate(ground_truth, vector_search)

In [ ]:
# Q6. Tuning hybrid search
results = {}
for k in [1, 50, 100, 200]:
    metrics = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    results[k] = metrics["mrr"]

results